# Research Workflow and Evaluation Handoff

Load the prepared market/news outputs, plan the research request, retrieve evidence, generate an answer, and validate the interface consumed by the evaluation and memory services.

In [17]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = Path("..").resolve()
load_dotenv(project_root / ".env", override=True)
sys.path.insert(0, str(project_root / "src"))

from agent.graph import AgentGraph
import routing.model as model_module
from tools.notebook_adapter import NotebookResearchProvider

model_module = importlib.reload(model_module)
HuggingFaceAnswerGenerator = model_module.HuggingFaceAnswerGenerator

In [18]:
stock_ticker = os.getenv("STOCK_TICKER", "AAPL").strip().upper()
market_path = project_root / f"data/processed/market_{stock_ticker}.csv"
news_path = project_root / f"data/processed/news_{stock_ticker}.json"
if not market_path.exists() or not news_path.exists():
    raise FileNotFoundError("Run 01_market_news_collection.ipynb first.")

market_df = pd.read_csv(market_path)
news_df = pd.read_json(news_path)
company_name = os.getenv("COMPANY_NAME", stock_ticker)
data_provider = NotebookResearchProvider(stock_ticker, market_df, news_df, company_name)
print(f"Loaded {len(market_df):,} market rows and {len(news_df):,} news rows.")

Loaded 251 market rows and 50 news rows.


In [19]:
model_name = os.getenv("HF_MODEL", "HuggingFaceH4/zephyr-7b-beta")
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
answer_generator = None

if hf_token:
    from huggingface_hub import InferenceClient
    answer_generator = HuggingFaceAnswerGenerator(
        InferenceClient(token=hf_token),
        model=model_name,
    )
else:
    print("No HF_TOKEN found; using the deterministic answer generator.")

In [20]:
research_graph = AgentGraph(data_provider, answer_generator=answer_generator)
try:
    research_state = research_graph.run(
        query=f"Analyze {company_name} ({stock_ticker}) using financial performance, market behavior, recent news, and material risks.",
        ticker=stock_ticker,
    )
except Exception as error:
    message = str(error)
    if "model_not_supported" not in message and "provider" not in message:
        raise
    print("Hugging Face has no enabled provider for the configured model.")
    print("Continuing with the deterministic evidence-grounded response.")
    answer_generator = None
    research_graph = AgentGraph(data_provider)
    research_state = research_graph.run(
        query=f"Analyze {company_name} ({stock_ticker}) using financial performance, market behavior, recent news, and material risks.",
        ticker=stock_ticker,
    )

print(f"Completed tasks: {len(research_state.completed_tasks)}")
print(f"Evidence records: {len(research_state.evidence)}")
print(research_state.draft_report)

Hugging Face has no enabled provider for the configured model.
Continuing with the deterministic evidence-grounded response.
Completed tasks: 6
Evidence records: 50
Research request: Analyze AAPL (AAPL) using financial performance, market behavior, recent news, and material risks.

Financial data:
- ticker: AAPL
- company: AAPL
- observations: 251
- latest_close: 336.64
- period_return: 0.377598
- latest_volume: 37855000

Evidence: New Mac Mini Has Two Storage Downsides, Including Unremovable SSD (MacRumors)
As confirmed by the website "Expand Mac mini," NAND storage in the new Mac mini models with the M6 and M5 Pro chips is soldered to the logic board, whereas the previous models with the M4 and M4 Pro chips have removable storage that can be upgraded.





In o… As confirmed by the website "Expand Mac mini," NAND storage in the new Mac mini models with the M6 and M5 Pro chips is soldered to the logic board, whereas the previous models with the M4 and M4 Pro … [+2226 chars]

Evidence:

In [21]:
# The evaluation and memory services consume these three values.
evaluation_query = research_state.user_query
evaluation_context = research_state.metadata["context"]
evaluation_answer = research_graph.route(evaluation_query, evaluation_context)

assert isinstance(evaluation_query, str) and evaluation_query.strip()
assert isinstance(evaluation_context, str) and evaluation_context.strip()
assert isinstance(evaluation_answer, str) and evaluation_answer.strip()

handoff = {
    "query": evaluation_query,
    "answer": evaluation_answer,
    "context": evaluation_context,
}
print("Validated handoff fields:", sorted(handoff))
print("The output is compatible with ResearchRouterService and EvaluationService.")

Validated handoff fields: ['answer', 'context', 'query']
The output is compatible with ResearchRouterService and EvaluationService.
